AI RESEARCH PAPER ASSISTANT AND **SEGREGATOR** bold text

# 1. Environment Setup

In [1]:
!pip install datasets


## 1.1 Import Libraries

from datasets import load_dataset

In [2]:
from datasets import load_dataset
import pandas as pd

# 2. Data Loading and Preprocessing

from arxiv site(contains research papers and need to find which one is relevant)only searching by title in not relevant, title + extract sematic analysis is helpful for relevancy

https://huggingface.co/datasets/CShorten/ML-ArXiv-Papers

in dataset its 4 columns two unnamed and one title and other extract

## 2.1 Load Dataset

In [3]:
dataset=load_dataset("CShorten/ML-ArXiv-Papers",split='train')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [5]:
dataset[0]

{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

## 2.2 Initial Data Inspection and Cleaning

dataset is not proper (in form of key value )we have to frame it

In [6]:
df=pd.DataFrame(dataset)

In [7]:
df


,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [8]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [9]:
df=df[['title','abstract']]

In [10]:
df

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
117587,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


## 2.3 Sampling Data for Easier Processing

In [11]:
df.shape

(117592, 2)

title+abstract for this is much data is quite huge

sir taking 15000 dataset for easy processing but we can take full

In [12]:
df=df.head(15000)

In [13]:
df.shape

(15000, 2)

## 2.4 Combine Title and Abstract

for every dataset embedding is done,11700*384 (int type) byte is quite huge

In [14]:
df["paper_text"]=df["title"]+" "+df["abstract"]

/tmp/ipykernel_2862/529180721.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["title"]+" "+df["abstract"]


In [15]:
df.isnull().sum()

,0
title,0
abstract,0
paper_text,0


In [16]:
df[["paper_text"]].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


## 2.5 Text Cleaning

In [17]:
type(df[["paper_text"]])

pandas.core.frame.DataFrame

In [18]:
print(df["paper_text"].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

implementing sentence transformer

In [19]:

from sentence_transformers import SentenceTransformer

## 3.1 Load Sentence Transformer Model

In [20]:
model = SentenceTransformer("all-MiniLM-L6-v2")



## 3.2 Generate and Inspect Sample Embeddings

In [21]:
print(type(model))


<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


when we create embedding ,we have to do nlp preprocessing technique

stop word removal,tokenization

In [22]:
sample_text=df['paper_text'].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparam

we have to replace /n as transformer can get confused

In [23]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

/tmp/ipykernel_2862/2190359946.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
/tmp/ipykernel_2862/2190359946.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.strip()


# 3. Sentence Embeddings for Semantic Search

In [24]:
embedding=model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [25]:
embedding[:56]

array([-0.1315641 , -0.00678266, -0.00367612,  0.03265158,  0.11219642,
        0.01227267,  0.09816719, -0.0900523 ,  0.04231161, -0.01977348,
       -0.03308417,  0.07452948,  0.10632038, -0.02060429, -0.02052106,
        0.00169493,  0.07081953,  0.05854454, -0.11231912,  0.02082474,
        0.05692544,  0.0201578 ,  0.0258311 ,  0.0321703 ,  0.10513764,
       -0.09676763,  0.02700802, -0.0234509 , -0.04549678, -0.01013699,
       -0.01794855, -0.04814427,  0.01077652, -0.03759069,  0.01943481,
        0.03715189,  0.02967844,  0.04330941,  0.04373213,  0.03704866,
       -0.00182594,  0.00455183, -0.00799067,  0.03037368, -0.014378  ,
        0.03795147,  0.0595916 , -0.02583356, -0.06521576,  0.05900268,
       -0.02107134,  0.07359422, -0.05720106,  0.00294061,  0.00767515,
       -0.0333416 ], dtype=float32)

# 4. Similarity Search with FAISS

In [26]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())

In [27]:
print(sample_embedding.shape)

(5, 384)


## 4.1 Cosine Similarity Introduction

SUGGESTION LIKE SERACHING FOR AUTOMOBILE AND RESEARCH PAPER CONTAINS CONTENT OF CAR AND BEHICLES SO SEMANTICALLY THEY ARE SAME

COSINE SIMILARITY (A.B/|A|.|B|)

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
'''similarity=cosine_similarity(sample_embedding[0],sample_embedding[0])'''

'similarity=cosine_similarity(sample_embedding[0],sample_embedding[0])'

cosine similarity expect 2d data,sample_embedding[0] has single embedding with [,384] but i should have one row and all names

reshape[1,-1] will take 1 row and adjust column to toal shapes

## 4.2 Demonstrating Cosine Similarity

eg [5,6]-->[5,6*5]=[5,30][

In [30]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[0].reshape(1,-1))

In [31]:
print(similarity)

[[1.0000001]]


In [32]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))

In [33]:
print(similarity)

[[0.36625272]]


## 4.3 Generating Embeddings for Entire Dataset

In [34]:
for i in range(1,5):
  sin=similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))
  print(sin)

[[0.36625272]]
[[0.36625272]]
[[0.36625272]]
[[0.36625272]]


GENERATION OF ALL 15000 PAPERS

In [35]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")

Loading saved embeddings


## 4.4 FAISS Index Creation

In [36]:
print(embedding.shape)
print(type(embedding))

(384,)
<class 'numpy.ndarray'>


In [37]:
embedding.dtype

dtype('float32')

FAISS~Facebook AI similarity search
(Vector database)


create a structure which can ease the process of search

build search structure for speed

internal structure with index creation on the top of the embedding

In [38]:
!pip install faiss-cpu

In [39]:
import faiss

## 4.5 Build and Save FAISS Index

In [40]:
import numpy as np

In [41]:
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    faiss.write_index(index, "paper_faiss.index")
    print("FAISS index saved successfully!")

Loading existing FAISS index


## 4.6 Testing the FAISS Index

for creating index first we normalized embedding,faiss creates structure flat creates same (neither compressed nor approxiamation),stores all the vectors,IP stands for innner product of all embeddings

above is just a function,but no passing of embedding

In [42]:
print(index.ntotal)

15000


In [43]:
query="deeep learning for medical image analysis"

In [44]:
query_embedding=model.encode(query)

In [45]:
query_embedding.shape

(384,)

In [46]:
query_embedding.reshape(1,-1)

array([[-8.91347881e-03,  4.24088500e-02,  3.96373235e-02,
        -8.75462294e-02, -4.33073239e-03, -9.53942016e-02,
         1.32165462e-01,  2.11447030e-02,  1.33173913e-02,
        -2.14715768e-03, -3.48479562e-02, -3.11603323e-02,
         2.90877558e-03,  5.49279116e-02, -8.22757334e-02,
        -1.32649997e-02, -1.17033020e-01, -1.47760464e-02,
        -3.85108143e-02,  5.51063344e-02, -5.35834655e-02,
         3.18396762e-02, -5.36726043e-02, -3.92478444e-02,
         3.43137011e-02,  2.04335582e-02,  6.49864376e-02,
         8.85641202e-03,  3.40326056e-02, -5.88501580e-02,
         4.21522669e-02,  3.36007308e-03, -5.37609216e-03,
        -1.55146541e-02,  2.37983558e-02,  1.27095738e-02,
         2.42141355e-02,  5.71320504e-02, -3.86737250e-02,
         1.07356057e-01,  3.09558789e-04,  6.11782297e-02,
        -3.49811949e-02, -2.42499020e-02,  7.20748976e-02,
        -1.52707947e-02, -2.86656488e-02, -5.77483997e-02,
         3.77038755e-02,  5.24876490e-02, -2.51735039e-0

## 4.7 Define Search Function

In [47]:
query_embedding.shape

(384,)

In [48]:
query_embedding_final=model.encode([query])
query_embedding_final.shape

(1, 384)

In [49]:
faiss.normalize_L2(query_embedding_final)


In [50]:
D,I=index.search(query_embedding_final,5)
print(D)
print(I)

[[0.5202042  0.51973665 0.504279   0.5003392  0.49550495]]
[[10466  5544 13730  8852  7426]]


this will give similarity score and seach index (give relevant 5 researches)

In [51]:
df.iloc[10466]["title"]

'A Perspective on Deep Imaging'

In [52]:
df.iloc[11873]["title"]

'Classification of MRI data using Deep Learning and Gaussian\n  Process-based Model Selection'

here mri means semantic meaning gapp reduce due sentence transformers

In [53]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Title",df.iloc[idx]["abstract"][:500])
    print()

# 5. Summarization with BART

In [54]:
search_paper("deep learning for medical image analysis")

Similarity score 0.6807244
Title A Perspective on Deep Imaging
Title   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity score 0.67092204
Title Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Title   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a large

BART (bidirectional transformer with both encoder and decoder)

## 5.1 Load Summarization Pipeline

In [55]:
!pip install transformers==4.51.3

In [56]:
from transformers import pipeline

In [57]:
summarizer=pipeline("summarization",model="facebook/bart-large-cnn")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


## 5.2 Test Summarization

In [58]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [59]:
summary=summarizer

In [60]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]


In [61]:
type(summary)

list

In [62]:
type(summary[0])

dict

In [63]:
summary[0]["summary_text"]

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

## 5.3 Integrate Summarization into Search

In [64]:
for score,idx in zip(D[0],I[0]):
  print("Similarity score",score)
  print("Title",df.iloc[idx]["title"])
  print("Abstract",df.iloc[idx]["abstract"][:500])
  summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
  print (summary)
  print(summary[0]["summary_text"])
  print()

Similarity score 0.5202042
Title A Perspective on Deep Imaging
Abstract   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance
[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]
The combination of tomographic imaging and deep learning, or mac

# 6. Keyword Extraction with KeyBERT

for important keyword extraction,using KEYBERT algoritm

KEYBERT has count vectorizer which remove stopwords

cosine similarity to all embedding and give word accordingly in ascending or descending order

## 6.1 Install and Load KeyBERT

In [65]:
def search_and_summarize (query, k=5) :
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding, k)
  for score, idx in zip(D[0],I[0]):
    print ("Similarity score", score)
    print("Title",df.iloc[idx]["title"])
    print("Title",df.iloc[idx]["abstract"][:500])
    print()
    summary=summarizer(df.iloc[idx]["abstract"], max_length=120,min_length=40,do_sample=False)
    print(summary)
    print (summary[0]["summary_text"])
    print()

https://huggingface.co/learn/llm-course/en/chapter1/5

https://maartengr.github.io/KeyBERT/guides/countvectorizer.html

In [66]:
search_and_summarize("Deep Imaging in medical learning",k=5)

Similarity score 0.6995392
Title A Perspective on Deep Imaging
Title   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]
The combination of tomographic imaging and deep learning, or machi

keybert will take contextual meaning important words ,not repeating keywords

In [67]:
!pip install keybert==0.8.5

In [68]:
from keybert import KeyBERT

## 6.2 Initialize KeyBERT Model

In [69]:
kw_model=KeyBERT()

In [70]:
type(kw_model)

keybert._model.KeyBERT

## 6.3 Test Keyword Extraction

automatically taking sentence trnasformer as model default as its just an algorithm and require a tranformer

In [71]:
text=df.iloc[10466]['abstract']
keywords=kw_model.extract_keywords(text)

In [72]:
print(keywords)

[('imaging', 0.4528), ('tomographic', 0.4488), ('reconstruction', 0.3623), ('deep', 0.3003), ('learning', 0.2622)]


In [73]:
print(type(keywords))

<class 'list'>


In [74]:
print(type(keywords[0]))

<class 'tuple'>


## 6.4 Integrate Keyword Extraction into Search and Summarize Function

In [75]:
kw_model.extract_keywords(text,keyphrase_ngram_range=(1,3),stop_words="english")

[('tomographic imaging deep', 0.6704),
 ('imaging deep learning', 0.6543),
 ('learning medical imaging', 0.6041),
 ('imaging deep', 0.5919),
 ('medical imaging', 0.5281)]

In [76]:
print(keywords)

[('imaging', 0.4528), ('tomographic', 0.4488), ('reconstruction', 0.3623), ('deep', 0.3003), ('learning', 0.2622)]


In [77]:
def search_and_summarize (query, k=5) :
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding, k)
  for score, idx in zip(D[0],I[0]):
    print ("Similarity score", score)
    print("Title",df.iloc[idx]["title"])
    print("Title",df.iloc[idx]["abstract"][:500])
    print()
    summary=summarizer(df.iloc[idx]["abstract"], max_length=120,min_length=40,do_sample=False)
    print(summary)
    print (summary[0]["summary_text"])
    print()
    kw_model.extract_keywords(text,keyphrase_ngram_range=(1,3),stop_words="english")
    print(keywords)
    for k,s in keywords:
      print(k)


# 7. Named Entity Recognition (NER) with GLiNER

In [78]:
search_and_summarize("Deep Imaging in medical learning",k=5)

Similarity score 0.6995392
Title A Perspective on Deep Imaging
Title   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]
The combination of tomographic imaging and deep learning, or machi

## 7.1 Langchain Tools Integration (Attempt)

more inclusion NER (named entity relationship)

In [79]:
pip install langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [80]:
from langchain_core.tools import tool

@tool gives additonal feature to function without

## 7.2 Langchain HuggingFace Integration

In [81]:
@tool
def search_paper(quert:str)->str:
  "Search  research paper from the faiss database and return the most relevant result"


In [82]:
pip install langchain-huggingface

In [83]:
from langchain_huggingface import HuggingFacePipeline

In [84]:
llm=HuggingFacePipeline(pipeline=summarizer)

## 7.3 Named Entity Recognition with GLiNER

we pipeline our code in hugging face so that automation can be done.


In [85]:
llm

HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, pipeline=<transformers.pipelines.text2text_generation.SummarizationPipeline object at 0x7d4b0a68d2b0>, model_id='facebook/bart-large-cnn')

In [86]:
response=llm.invoke("Breast cancer screening and longitudinal monitoring require imaging technologies that are portable, operator-independent, and suitable for frequent use, a capability not fully met by mammography or conventional ultrasound. We present a three-dimensional (3D) portable ultrasound system for real-time examination (3D PURE) that overcomes key limitations in volumetric breast imaging through advances in transducer design, acoustic materials, and adaptive beamforming. A box-array design incorporating a corner-gap offset geometry suppresses peak crosstalk (by 3.73 dB at the corner-most element), prevents preamplifier saturation, and supports higher transmit voltages (up to 24 V). A custom flowable backing layer (impedance 6.12 MRayl; attenuation 7.56 dB mm−1 MHz−1) integrates around fragile wirebonds, reduces inter-element crosstalk by ~4.5 dB throughout the array, and improves axial and lateral/elevational resolutions by ~200 µm and ~70 µm, respectively. Layered Aberration-Correction Reconstruction (LACR), an adaptive 3D beamformer, compensates for heterogeneous speed-of-sound (SoS) in the breast, reducing depth localization error by 2 mm and aberration defocusing by 70 µm on average at a 5 cm depth. Nine of the ten participants in an in vitro study showed improved microtarget detection efficiency with 3D PURE relative to a conventional 2D system (p = 0.0215), analogous to detection of microcalcifications. These results, combined with in vivo imaging validation of various phenotypes, highlight the potential of 3D PURE for reliable breast imaging. Furthermore, a vision-guided computer interface, MyFUS, ensures self-guided, user-friendly, and operator-independent probe positioning for longitudinal monitoring.")

In [87]:
print(response)

3D PURE overcomes key limitations in volumetric breast imaging through advances in transducer design, acoustic materials, and adaptive beamforming. A box-array design incorporating a corner-gap offset geometry suppresses peak crosstalk. Layered Aberration-Correction Reconstruction (LACR) compensates for heterogeneous speed-of-sound in the breast.


In [88]:
print(type(response))

<class 'str'>


In [89]:
!pip install gliner
from gliner import GLiNER

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 47.1 MB/s eta 0:00:00


In [90]:
!pip install flashdeberta -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 7.2 MB/s eta 0:00:00


In [91]:
os.environ["USE_FLASHDEBERTA"] = "1"

## 7.4 Define NER Function

In [92]:
gliner_model = GLiNER.from_pretrained("knowledgator/gliner-decoder-base-v1.0")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

In [93]:
def NER_papers(abstract_list,result):
    labels = ["label"]

    for abstract in abstract_list:
        entities = gliner_model.predict_entities(abstract, labels, threshold=0.3, num_gen_sequences=1)

        extracted_data = []
        for entity in entities:
            extracted_data.append({
                "text": entity["text"],
                "label": entity["label"],
                "score": entity["score"]
            })
        result.append(extracted_data)
    return result

## 7.5 Define Search Function for Abstract List

In [94]:
def search_paper12(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  abstract=[]
  for score,idx in zip(D[0],I[0]):
    abstract.append(df.iloc[idx]["abstract"])
  return abstract

## 7.6 Execute Search and NER

In [95]:
abstract_list=search_paper12("Deep Imaging in medical learning",k=5)

In [96]:
for i in abstract_list:
  print(i)
  print()

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.


  Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a large set of labeled
natural images. However, the substant

In [97]:
result=[]

In [98]:
result=NER_papers(abstract_list,result)
show_progress_bar=True

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [99]:
type(result)

list

In [100]:
for i in result:
  print(i)

[{'text': 'tomographic imaging', 'label': 'label', 'score': 0.9443729519844055}, {'text': 'deep learning', 'label': 'label', 'score': 0.9661700129508972}, {'text': 'machine learning', 'label': 'label', 'score': 0.955564558506012}, {'text': 'image analysis', 'label': 'label', 'score': 0.8078495264053345}, {'text': 'image\nreconstruction', 'label': 'label', 'score': 0.834284245967865}, {'text': 'perspective article', 'label': 'label', 'score': 0.4489283263683319}, {'text': 'medical imaging', 'label': 'label', 'score': 0.8953460454940796}, {'text': 'image\nreconstruction', 'label': 'label', 'score': 0.7804543375968933}, {'text': 'domain knowledge', 'label': 'label', 'score': 0.5012676119804382}, {'text': 'big data', 'label': 'label', 'score': 0.8758136630058289}, {'text': 'innovative\napproaches', 'label': 'label', 'score': 0.37934067845344543}, {'text': 'image reconstruction', 'label': 'label', 'score': 0.859889566898346}, {'text': 'clinical and\npreclinical applications', 'label': 'labe

## 7.7 Spacy Integration

In [101]:
!pip install spacy

In [102]:
import spacy

In [103]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 26.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [104]:
nlp = spacy.load('en_core_web_sm')

## 7.8 Define SpaCy NER Function

In [105]:
from collections import defaultdict

def NER_papers_spacy(abstract_list):
    spacy_results = []
    for abstract in abstract_list:
        doc = nlp(abstract)
        categorized_entities = defaultdict(list)
        for ent in doc.ents:
            categorized_entities[ent.label_].append(ent.text)
        spacy_results.append(dict(categorized_entities))
    return spacy_results

## 7.9 Demonstrate SpaCy NER Function

In [106]:
spacy_extracted_entities = NER_papers_spacy(abstract_list)
for doc_entities in spacy_extracted_entities:
    if doc_entities:
        print(doc_entities)
    else:
        print("No entities found for this abstract.")
    print()

No entities found for this abstract.

{'ORG': ['CNN', 'CNN', 'CNN', 'CNN', 'CNN'], 'CARDINAL': ['4', '3', '3', '1', '2', '3', '4']}

{'ORG': ['AlexNet', 'Gaussian Processes and Probability'], 'ORDINAL': ['first'], 'CARDINAL': ['20\\%']}

{'CARDINAL': ['886,000']}

{'ORG': ['Computer\nAided Diagnosis', 'CAD', 'CNN', 'Computed Tomography', 'CT', 'CNN']}



In [120]:
!pip install -qU langchain_groq

In [123]:
from langchain_groq import ChatGroq

In [114]:
from google.colab import userdata

In [145]:
api_key = userdata.get("groq_api_key")

In [146]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key,
    temperature=0.7,
    timeout=30,
    max_tokens=1000
)

In [165]:
responses=llm.invoke("Hello, who are you?")

In [166]:
responses

AIMessage(content="Hello. I'm an artificial intelligence model, a computer program designed to simulate conversations and answer questions to the best of my knowledge. I'm here to help and provide information on a wide range of topics, from science and history to entertainment and culture. I can also assist with tasks such as language translation, text summarization, and more.\n\nI don't have a personal identity or emotions, but I'm designed to be helpful and engaging. I'm constantly learning and updating my knowledge base to provide more accurate and up-to-date information.\n\nWhat would you like to talk about or ask?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 41, 'total_tokens': 159, 'completion_time': 0.165710869, 'completion_tokens_details': None, 'prompt_time': 0.002592886, 'prompt_tokens_details': None, 'queue_time': 0.051152476, 'total_time': 0.168303755}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_0

In [167]:
type(responses)

langchain_core.messages.ai.AIMessage

In [132]:
from langchain_core.tools import tool

In [150]:
@tool
def search_and_summarize(query: str, k: int = 5) -> str:
    """
    Search research papers from the FAISS database,
    retrieve the top-k most similar papers,
    summarize each paper using BART,
    and return the results.
    """

    query_embedding = model.encode([query])


    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    result = ""
    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

        paper = df.iloc[idx]


        summary = summarizer(
            paper["abstract"],
            max_length=60, # Reduced max_length
            min_length=20, # Reduced min_length
            do_sample=False
        )[0]["summary_text"]

        result += f"Rank: {rank}\n"
        result += f"Similarity Score: {round(float(score),4)}\n"
        result += f"Title: {paper['title']}\n\n"

        result += "Abstract:\n"
        result += paper["abstract"][:500] + "...\n\n" # Truncated abstract

        result += summary + "\n\n"
    return result

In [151]:
tools = [search_and_summarize]

In [152]:
from langchain.agents import create_agent

In [159]:
agent= create_agent(
    model=llm,
    tools=tools
)

In [173]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find the top 3 research papers on Vision Transformer."
            }
        ]
    }
)

In [171]:
responses.content

"Hello. I'm an artificial intelligence model, a computer program designed to simulate conversations and answer questions to the best of my knowledge. I'm here to help and provide information on a wide range of topics, from science and history to entertainment and culture. I can also assist with tasks such as language translation, text summarization, and more.\n\nI don't have a personal identity or emotions, but I'm designed to be helpful and engaging. I'm constantly learning and updating my knowledge base to provide more accurate and up-to-date information.\n\nWhat would you like to talk about or ask?"

In [174]:
response


{'messages': [HumanMessage(content='Find the top 3 research papers on Vision Transformer.', additional_kwargs={}, response_metadata={}, id='61909f86-b053-45b5-b34e-15577d506cc3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ahm35fwa5', 'function': {'arguments': '{"k":3,"query":"Vision Transformer"}', 'name': 'search_and_summarize'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 290, 'total_tokens': 314, 'completion_time': 0.033067153, 'completion_tokens_details': None, 'prompt_time': 0.022570726, 'prompt_tokens_details': None, 'queue_time': 0.006109852, 'total_time': 0.055637879}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4165-7fcc-7363-8d46-58dbb52c0db3-0', tool_calls=[{'name': 'search_and_summarize', 'args': {'k': 3, 'query': 'Vision Transformer'}, 'id': '

In [175]:
print(type(response))


<class 'dict'>


In [176]:

response["messages"][0]


HumanMessage(content='Find the top 3 research papers on Vision Transformer.', additional_kwargs={}, response_metadata={}, id='61909f86-b053-45b5-b34e-15577d506cc3')

In [177]:
response["messages"][1]


AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ahm35fwa5', 'function': {'arguments': '{"k":3,"query":"Vision Transformer"}', 'name': 'search_and_summarize'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 290, 'total_tokens': 314, 'completion_time': 0.033067153, 'completion_tokens_details': None, 'prompt_time': 0.022570726, 'prompt_tokens_details': None, 'queue_time': 0.006109852, 'total_time': 0.055637879}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4165-7fcc-7363-8d46-58dbb52c0db3-0', tool_calls=[{'name': 'search_and_summarize', 'args': {'k': 3, 'query': 'Vision Transformer'}, 'id': 'ahm35fwa5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 24, 'total_tokens': 314})

In [178]:
response["messages"][2]

ToolMessage(content='Rank: 1\nSimilarity Score: 0.4415\nTitle: Dense Transformer Networks\n\nAbstract:\n  The key idea of current deep learning methods for dense prediction is to\napply a model on a regular patch centered on each pixel to make pixel-wise\npredictions. These methods are limited in the sense that the patches are\ndetermined by network architecture instead of learned from data. In this work,\nwe propose the dense transformer networks, which can learn the shapes and sizes\nof patches from data. The dense transformer networks employ an encoder-decoder\narchitecture, and a pair of dense transformer modules are inserted into each of\nthe encoder and decoder paths. The novelty of this work is that we provide\ntechnical solutions for learning the shapes and sizes of patches from data and\nefficiently restoring the spatial correspondence required for dense prediction.\nThe proposed dense transformer modules are differentiable, thus the entire\nnetwork can be trained. We apply th